# Sampling synthetic data for fine-tuning

### Imports & Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import colorsys
import json
import os
import pathlib
from collections import defaultdict
from typing import Any

import aesthetics as aes
import pandas as pd
import pyrootutils
from datasets import Dataset
from transformers import AutoTokenizer

from formal_gym import prompt as fg_prompt

In [ ]:
# type aliases
Path = pathlib.Path

PROJECT_ROOT: Path = pyrootutils.find_root(
    search_from=os.path.abspath(""), indicator=".project-root"
)

## Load Annotated Data

In [ ]:
response_df_file = PROJECT_ROOT / "data" / "response_df.feather"

response_df = pd.read_feather(response_df_file)

assert len(response_df) == 808445, "Incomplete response_df loaded!"
assert not response_df.isnull().values.any(), "response_df contains null values!"

response_df.info()

In [ ]:
# filter to only include rows where 'model' is gpt-4.1 and "correct" is True
response_df = response_df.query("model == 'gpt-4.1' and correct == True").reset_index(
    drop=True
)

response_df.info()

In [ ]:
# Load grammar files and add to dataframe

grammar_files: dict[str, str] = {}
for grammar_file in (PROJECT_ROOT / "data" / "grammars").rglob("*.cfg"):
    grammar_name = grammar_file.parent.name
    grammar = grammar_file.read_text()
    grammar_files[grammar_name] = grammar

response_df["grammar"] = response_df["grammar_file"].map(lambda x: grammar_files[x])

In [ ]:
# drop the n_shots, batch_id, reasoning_tokens, model_type columns, if they are present
response_df = response_df.drop(
    columns=[
        col
        for col in ["n_shots", "batch_id", "reasoning_tokens", "model_type", "correct"]
        if col in response_df.columns
    ]
)

In [ ]:
response_df.info()

In [ ]:
# response_df has columns `grammar` and `sample`, we can use these to construct the prompt using fg_prompt.basic_prompt(grammar_str: str, sample: str, shots: list[str]) -> str. we'll pass in an empty list for shots.

response_df["prompt"] = response_df.apply(
    lambda row: fg_prompt.basic_prompt(
        grammar_str=row["grammar"],
        sample=row["sample"],
        shots={"positive": [], "negative": []},
    ),
    axis=1,
)

# save to feather file
synthetic_data_file = (
    PROJECT_ROOT / "notebooks" / "data" / "synthetic" / "ft_data.feather"
)
response_df.to_feather(synthetic_data_file)

## Format for Gemma FT

In [ ]:
def format_gemma_chat(row):
    return (
        f"<start_of_turn>user\n{row['prompt']}<end_of_turn>\n"
        f"<start_of_turn>model\n{row['model_response']}<end_of_turn>\n"
    )


response_df["text"] = response_df.apply(format_gemma_chat, axis=1)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
tokenizer.pad_token = tokenizer.eos_token


def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=2048,
    )


dataset = Dataset.from_pandas(response_df[["text"]])
tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
)

In [ ]:
dataset = Dataset.from_pandas(response_df[["prompt", "model_response"]])


def preprocess_fn(example):
    return {
        "prompt": [{"role": "user", "content": example["prompt"]}],
        "completion": [
            {"role": "assistant", "content": f"{example['model_response']}"}
        ],
    }


dataset = dataset.map(preprocess_fn, remove_columns=["prompt", "model_response"])